# 🔍 InFi-Check — Eval & Reference Generation Pipeline

Notebook này thực hiện **Bước 4** trong pipeline InFi-Check:
```
new_summary/*_summary.txt  →  [eval_and_reference_gen]  →  new_supported_summary/ + new_reference/
```

| Bước | File | Mô tả |
|------|------|-------|
| 1 | `dataset_*.py` | Làm sạch data |
| 2 | ✅ | Upload document lên Drive |
| 3 | ✅ `summary_gen_pipeline.ipynb` | Sinh tóm tắt |
| **4** | **`eval_and_reference_gen_pipeline.ipynb`** | **← Notebook này** |
| 5 | `structured_dataset_gen.py` | Sinh error data |
| 6 | `prepare_dataset_base.py` | Xuất JSONL |

**Notebook này làm 2 việc cho mỗi summary (đã cập nhật):**
1. **find_support** — dùng `gpt-4o-mini` tìm câu nào trong document hỗ trợ từng câu tóm tắt
2. **critics (3 model vote)** — dùng `gpt-4o-mini` + `Qwen-72B` + `Llama-70B` kiểm tra từng câu; nếu sai thì tự sửa

Output cuối:
- `new_supported_summary/<name>_supported_summary.txt` — tóm tắt đã được xác nhận
- `new_reference/<name>_ref.json` — câu hỗ trợ từ document cho từng câu tóm tắt

## 1. Mount Google Drive & kiểm tra cấu trúc thư mục

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ✏️ CHỈNH đường dẫn nếu cần
PROJECT_ROOT     = '/content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset'
BASE_DOC_FOLDER  = '/content/drive/MyDrive/Dataset NLP'   # folder gốc chứa các category

SUMMARY_ROOT             = os.path.join(PROJECT_ROOT, 'new_summary')
SUPPORTED_SUMMARY_FOLDER = os.path.join(PROJECT_ROOT, 'new_supported_summary')
REFERENCE_FOLDER         = os.path.join(PROJECT_ROOT, 'new_reference')
PROMPT_FOLDER            = os.path.join(PROJECT_ROOT, '..', 'summary_eval_prompt')

os.makedirs(SUPPORTED_SUMMARY_FOLDER, exist_ok=True)
os.makedirs(REFERENCE_FOLDER,         exist_ok=True)

# Scan tất cả summary theo subfolder: (category, filename, full_path)
all_summary_pairs = []  # list of (category, fname, full_path)
for cat in sorted(os.listdir(SUMMARY_ROOT)):
    cat_path = os.path.join(SUMMARY_ROOT, cat)
    if not os.path.isdir(cat_path):
        continue
    for f in sorted(os.listdir(cat_path)):
        if f.endswith('.txt'):
            all_summary_pairs.append((cat, f, os.path.join(cat_path, f)))

done_files = [f for f in os.listdir(SUPPORTED_SUMMARY_FOLDER) if f.endswith('.txt')]

print(f'📂 Summary root       : {SUMMARY_ROOT}')
print(f'📂 Doc root           : {BASE_DOC_FOLDER}')
print(f'📂 Supported summary  : {SUPPORTED_SUMMARY_FOLDER}')
print(f'📂 Reference          : {REFERENCE_FOLDER}')
print(f'📄 Tổng summary       : {len(all_summary_pairs)} files')
print(f'✅ Đã xử lý xong      : {len(done_files)} files')
print(f'⏳ Còn lại            : {len(all_summary_pairs) - len(done_files)} files')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Summary root       : /content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset/new_summary
📂 Doc root           : /content/drive/MyDrive/Dataset NLP
📂 Supported summary  : /content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset/new_supported_summary
📂 Reference          : /content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset/new_reference
📄 Tổng summary       : 685 files
✅ Đã xử lý xong      : 647 files
⏳ Còn lại            : 38 files


## 2. Cài đặt thư viện

In [ ]:
!pip install -q openai

## 3. Cấu hình API keys

> 🔑 **Khuyên dùng Colab Secrets** (`Tools → Secrets`) thay vì paste key trực tiếp.
>
> Cần 3 secrets:
> - `OPENAI_API_KEY`  — dùng cho `gpt-4o-mini` (find_support + critics revise)
> - `OPENROUTER_API_KEY` — dùng cho `Qwen-72B` (critics)
> - `GROQ_API_KEY`    — dùng cho `Llama-70B` (critics)

In [ ]:
from google.colab import userdata

# --- Fetch keys from Colab Secrets ---
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# Support multiple OpenRouter keys
OPENROUTER_KEYS = []
for k in ['OPENROUTER_API_KEY', 'OPENROUTER_API_KEY1', 'OPENROUTER_API_KEY2','OPENROUTER_API_KEY3','OPENROUTER_API_KEY4','OPENROUTER_API_KEY5']:
    val = userdata.get(k)
    if val:
        OPENROUTER_KEYS.append(val)

# Support multiple Groq keys for rotation
GROQ_KEYS = []
for k in ['GROQ_API_KEY', 'GROQ_API_KEY1', 'GROQ_API_KEY2','GROQ_API_KEY3','GROQ_API_KEY4','GROQ_API_KEY5','GROQ_API_KEY6','GROQ_API_KEY7','GROQ_API_KEY8','GROQ_API_KEY9','GROQ_API_KEY10','GROQ_API_KEY11','GROQ_API_KEY12','GROQ_API_KEY13','GROQ_API_KEY14', 'GROQ_API_KEY15',
    'GROQ_API_KEY16', 'GROQ_API_KEY17', 'GROQ_API_KEY18', 'GROQ_API_KEY19','GROQ_API_KEY20', 'GROQ_API_KEY21', 'GROQ_API_KEY22', 'GROQ_API_KEY23','GROQ_API_KEY24']:
    val = userdata.get(k)
    if val:
        GROQ_KEYS.append(val)

def _mask(k):
    return f'{k[:8]}...{k[-4:]}' if k and len(k) > 12 else '❌ CHƯA CÓ'

print(f'OpenAI key      : {_mask(OPENAI_API_KEY)}')
print(f'OpenRouter keys : {len(OPENROUTER_KEYS)}')
for i, ok in enumerate(OPENROUTER_KEYS):
    print(f'  - Key {i+1}: {_mask(ok)}')
print(f'Groq keys found : {len(GROQ_KEYS)}')
for i, gk in enumerate(GROQ_KEYS):
    print(f'  - Key {i+1}: {_mask(gk)}')

OpenAI key      : sk-proj-...wfUA
OpenRouter keys : 6
  - Key 1: sk-or-v1...fa65
  - Key 2: sk-or-v1...faa4
  - Key 3: sk-or-v1...f598
  - Key 4: sk-or-v1...42c9
  - Key 5: sk-or-v1...823c
  - Key 6: sk-or-v1...b9e2
Groq keys found : 25
  - Key 1: gsk_qOrn...Lcdk
  - Key 2: gsk_qxw4...75sN
  - Key 3: gsk_X2h8...VzDY
  - Key 4: gsk_2awP...WODI
  - Key 5: gsk_2iby...Js1i
  - Key 6: gsk_UWBj...uGXp
  - Key 7: gsk_RyE8...NehD
  - Key 8: gsk_lfde...7t6e
  - Key 9: sk-or-v1...917d
  - Key 10: gsk_rvbg...RMJL
  - Key 11: gsk_V6uX...dh80
  - Key 12: gsk_6R0n...Hv84
  - Key 13: gsk_tIr1...0Wjj
  - Key 14: gsk_7QCM...S8cT
  - Key 15: gsk_CUG2...a36k
  - Key 16: gsk_dEwO...l3kH
  - Key 17: gsk_x8t9...gjvU
  - Key 18: gsk_3cY8...hPUU
  - Key 19: gsk_cbfF...HbB6
  - Key 20: gsk_ovb3...LjP7
  - Key 21: gsk_3Dac...Rx18
  - Key 22: gsk_uEil...jC9I
  - Key 23: gsk_iyAL...eYTU
  - Key 24: GROQ_API...EY17
  - Key 25: gsk_Ms6r...D1eM


## 4. Đọc các file prompt từ Drive

Đảm bảo thư mục `summary_eval_prompt/` đã được upload lên Drive và chứa đủ 6 file:
- `find_support.txt`
- `find_support_format.txt`
- `critics.txt`
- `critics_format.txt`
- `critics_with_revise.txt`
- `critics_with_revise_format.txt`

In [ ]:
def load_prompt(filename: str) -> str:
    path = os.path.join(PROMPT_FOLDER, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f'❌ Không tìm thấy prompt file: {path}')
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

FIND_SUPPORT_PROMPT          = load_prompt('find_support.txt')
FIND_SUPPORT_FORMAT_PROMPT   = load_prompt('find_support_format.txt')
CRITICS_PROMPT               = load_prompt('critics.txt')
CRITICS_FORMAT_PROMPT        = load_prompt('critics_format.txt')
CRITICS_REVISE_PROMPT        = load_prompt('critics_with_revise.txt')
CRITICS_REVISE_FORMAT_PROMPT = load_prompt('critics_with_revise_format.txt')

print('✅ Đã load xong 6 file prompt:')
for name in [
    'find_support.txt', 'find_support_format.txt',
    'critics.txt', 'critics_format.txt',
    'critics_with_revise.txt', 'critics_with_revise_format.txt'
]:
    print(f'   📄 {name}')

✅ Đã load xong 6 file prompt:
   📄 find_support.txt
   📄 find_support_format.txt
   📄 critics.txt
   📄 critics_format.txt
   📄 critics_with_revise.txt
   📄 critics_with_revise_format.txt


## 5. Khởi tạo clients & định nghĩa hàm tiện ích

In [ ]:
import re
import time
import json
from openai import OpenAI, BadRequestError
from difflib import SequenceMatcher
from requests.exceptions import RequestException

# ── Clients ────────────────────────────────────────────────────────────
openai_client = OpenAI(
    api_key  = OPENAI_API_KEY,
    base_url = 'https://api.openai.com/v1'
)

initial_or_key = OPENROUTER_KEYS[0] if OPENROUTER_KEYS else 'MISSING_KEY'
qwen_client = OpenAI(
    api_key  = initial_or_key,
    base_url = 'https://openrouter.ai/api/v1'
)

initial_groq_key = GROQ_KEYS[0] if GROQ_KEYS else 'MISSING_KEY'
groq_client = OpenAI(
    api_key  = initial_groq_key,
    base_url = 'https://api.groq.com/openai/v1'
)

FIND_SUPPORT_CLIENT = openai_client
FIND_SUPPORT_MODEL  = 'gpt-4o-mini'

MODEL_LIST = [
    {'name': 'gpt-4o-mini',                'client': openai_client, 'revise': True},
    {'name': 'qwen/qwen-2.5-72b-instruct', 'client': qwen_client,   'revise': False},
    {'name': 'llama-3.3-70b-versatile',    'client': groq_client,   'revise': False},
]

# ── Source description ─────────────────────────────────────────────────
def get_source_description(category: str) -> str:
    return 'Vietnamese news article'

# ── Tiện ích ───────────────────────────────────────────────────────────
def sent_tokenize_vi(text: str) -> list:
    """Tách câu cho cả tiếng Việt lẫn tiếng Anh (thay nltk)."""
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

def similarity(s1: str, s2: str) -> float:
    return SequenceMatcher(None, s1, s2).ratio()

def clean_llm_output(raw: str) -> str:
    """Loại bỏ markdown code fence trước khi eval()."""
    raw = raw.strip()
    for fence in ('```python', '```json', '```'):
        if raw.startswith(fence):
            raw = raw[len(fence):]
    if raw.endswith('```'):
        raw = raw[:-3]
    return raw.strip()

print('✅ Clients & hàm tiện ích đã sẵn sàng')

✅ Clients & hàm tiện ích đã sẵn sàng


## 6. Định nghĩa hàm `eval_summary`

In [ ]:
def eval_summary(document: str, summary: str, source_description: str):
    find_support_result = []; eval_result = []; errors = []

    # ── Bước 1: find_support ──────────────────────────────────────────
    for attempt in range(5):
        try:
            completion = FIND_SUPPORT_CLIENT.chat.completions.create(
                model=FIND_SUPPORT_MODEL,
                messages=[{'role': 'system', 'content': 'Support finder.'}, {'role': 'user', 'content': f'{FIND_SUPPORT_PROMPT}\n\nDoc:\n{document}\nSumm:\n{summary}\n\n{FIND_SUPPORT_FORMAT_PROMPT}'}]
            )
            summary_sentences = sent_tokenize_vi(summary)
            find_support_result = eval(clean_llm_output(completion.choices[0].message.content))
            find_support_result = [{'document': ' '.join(r['sentences from the document']), 'summary sentence': r['summary sentence'], 'reference': r['sentences from the document']} for r in find_support_result]
            break
        except Exception as e: print(f'  [find_support attempt {attempt+1}] error: {e}')

    if not find_support_result: return 'CANNOT PARSE RESULT', []

    # ── Bước 2: critics (3 model vote) ───────────────────────────────
    for r in find_support_result:
        revision = ''; supported_num = 0; response_num = 0; revise_count = 0
        while revise_count <= 3:
            for model_cfg in MODEL_LIST:
                retries = 0
                while retries < 5:
                    try:
                        current_client = model_cfg['client']
                        # Rotate keys for Groq and OpenRouter
                        if 'groq' in model_cfg['name'].lower() and GROQ_KEYS:
                            current_client.api_key = GROQ_KEYS[retries % len(GROQ_KEYS)]
                        elif 'qwen' in model_cfg['name'].lower() and OPENROUTER_KEYS:
                            current_client.api_key = OPENROUTER_KEYS[retries % len(OPENROUTER_KEYS)]

                        prompt_text = CRITICS_REVISE_PROMPT if model_cfg['revise'] else CRITICS_PROMPT
                        fmt_text = CRITICS_REVISE_FORMAT_PROMPT if model_cfg['revise'] else CRITICS_FORMAT_PROMPT

                        comp = current_client.chat.completions.create(
                            model=model_cfg['name'],
                            messages=[{'role': 'system', 'content': 'Strict checker.'}, {'role': 'user', 'content': f'{prompt_text}\n\nDoc:\n{r["document"]}\nSumm:\n{r["summary sentence"]}\n{fmt_text}'}]
                        )
                        result = eval(clean_llm_output(comp.choices[0].message.content))
                        response_num += 1
                        if result['support or not'] == 'YES': supported_num += 1
                        elif model_cfg['revise']: revision = result.get('summary sentence that is supported', '')
                        break
                    except Exception as e:
                        err_msg = str(e).lower()
                        if '401' in err_msg or 'api_key' in err_msg:
                            print(f"  ⚠️ Model {model_cfg['name']} rotation attempt {retries+1}")
                        time.sleep(1)
                        retries += 1

            if supported_num > 0.5 * response_num:
                eval_result.append({'summary sentence': r['summary sentence'], 'reference': r['reference'], 'votes': f'{supported_num}/{response_num}'})
                break
            elif revision: r['summary sentence'] = revision; revision = ''; supported_num = 0; response_num = 0; revise_count += 1
            else: revise_count += 1
    return eval_result, errors

## 7. Kiểm tra nhanh với 1 file trước khi chạy toàn bộ

## 8. Chạy toàn bộ dataset

> ⚡ **Tip Colab timeout:** Nếu session hết giờ giữa chừng, chỉ cần **Run All** lại —  
> file đã xử lý sẽ tự động được bỏ qua nhờ cơ chế resume.
>
> ⚙️ Đặt `LIMIT = 50` để xử lý 50 file một lần nếu lo hết quota API.

In [ ]:
# ✏️ Đổi thành số nguyên (vd: 50) nếu muốn xử lý từng batch
LIMIT = None

stats     = {'done': 0, 'skipped': 0, 'error': 0}
processed = 0

print(f'Tổng {len(all_summary_pairs)} file cần xử lý.\n')

for category, summary_name, summary_path in all_summary_pairs:
    if LIMIT is not None and processed >= LIMIT:
        print(f'\n🛑 Đã đạt giới hạn {LIMIT} file.')
        break

    # ── Resume: bỏ qua file đã có ─────────────────────────────────────
    doc_name         = summary_name.replace('_summary.txt', '')
    out_summary_path = os.path.join(
        SUPPORTED_SUMMARY_FOLDER,
        summary_name.replace('_summary', '_supported_summary')
    )
    if os.path.exists(out_summary_path):
        stats['skipped'] += 1
        continue

    # ── Tìm document gốc trong category folder ────────────────────────
    doc_file = os.path.join(BASE_DOC_FOLDER, category, f'{doc_name}.txt')
    if not os.path.exists(doc_file):
        print(f'[WARN] Không tìm thấy document: {doc_file}')
        stats['error'] += 1
        continue

    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = f.read().replace('"', "'").replace('\u201c', "'").replace('\u201d', "'")

    with open(doc_file, 'r', encoding='utf-8') as f:
        document = f.read().replace('"', "'").replace('\u201c', "'").replace('\u201d', "'")

    _words = document.split()
    if len(_words) > 800:
        document = ' '.join(_words[:800])

    # category chính là tên subfolder -> source description chính xác
    src = get_source_description(category)
    print(f'\n━━━ [{processed+1}] {doc_name}  ({src}) ━━━')

    # ── Đánh giá ──────────────────────────────────────────────────────
    eval_result, errors = eval_summary(document, summary, src)

    if not isinstance(eval_result, list):
        print(f'  ❌ Lỗi: {eval_result}')
        stats['error'] += 1
        continue

    # ── Lưu kết quả ───────────────────────────────────────────────────
    supported_summary = ' '.join([r['summary sentence'] for r in eval_result])
    reference_result  = {'find_support_result': eval_result, 'errors': errors}

    with open(out_summary_path, 'w', encoding='utf-8') as f:
        f.write(supported_summary)

    ref_path = os.path.join(REFERENCE_FOLDER, f'{doc_name}_ref.json')
    with open(ref_path, 'w', encoding='utf-8') as f:
        json.dump(reference_result, f, indent=4, ensure_ascii=False)

    print(f'  ✅ {len(eval_result)} câu  |  {os.path.basename(out_summary_path)}')
    stats['done'] += 1
    processed += 1

print(f'\n{"="*55}')
print(f'📊 Kết quả:')
print(f'   ✅ Xử lý thành công : {stats["done"]}')
print(f'   ⏭️  Đã có sẵn (skip) : {stats["skipped"]}')
print(f'   ❌ Lỗi              : {stats["error"]}')
remaining = len(all_summary_pairs) - stats['done'] - stats['skipped'] - stats['error']
print(f'   ⏳ Còn lại          : {remaining}')


Tổng 685 file cần xử lý.


📊 Kết quả:
   ✅ Xử lý thành công : 0
   ⏭️  Đã có sẵn (skip) : 685
   ❌ Lỗi              : 0
   ⏳ Còn lại          : 0


## 9. Xem trước kết quả

In [ ]:
import os
import re

def clean_name(filename):
    name = filename

    # bỏ _summary và (1)
    name = re.sub(r'_summary(\(\d+\))?', '', name)

    # bỏ .txt.txt -> .txt
    name = name.replace('.txt.txt', '.txt')

    return name

# test
print(clean_name("abc_summary(1).txt.txt"))
# -> abc.txt

abc.txt


In [ ]:
import random

done = [f for f in os.listdir(SUPPORTED_SUMMARY_FOLDER) if f.endswith('.txt')]
refs = [f for f in os.listdir(REFERENCE_FOLDER)         if f.endswith('.json')]

print(f'Supported summary: {len(done)} files')
print(f'Reference JSON   : {len(refs)} files\n')

# In ngẫu nhiên 3 cặp (summary + số câu reference)
for fname in random.sample(done, min(3, len(done))):
    with open(os.path.join(SUPPORTED_SUMMARY_FOLDER, fname), 'r', encoding='utf-8') as f:
        text = f.read()

    doc_name = fname.replace('_supported_summary.txt', '')
    ref_file = os.path.join(REFERENCE_FOLDER, f'{doc_name}_ref.json')
    n_sents  = 'N/A'
    if os.path.exists(ref_file):
        with open(ref_file, 'r', encoding='utf-8') as f:
            ref = json.load(f)
        n_sents = len(ref.get('find_support_result', []))

    print(f'📄 {fname}')
    print(f'   Số câu được xác nhận: {n_sents}')
    print(f'   Preview: {text[:200]}...')
    print()

Supported summary: 647 files
Reference JSON   : 647 files

📄 Nữ_trung_úy_thu_hút_với_loạt_ảnh_mặc_quân_phục_supported_summary.txt
   Số câu được xác nhận: 3
   Preview: Nguyễn Thị Thủy đang chuẩn bị diễn màn hợp xướng trong chương trình 'Đất nước trọn niềm vui' và hình ảnh của cô thu hút triệu lượt xem trên TikTok, Facebook vào tối 20/4. Nguyễn Thị Thủy còn có nghệ d...

📄 Tranh_cãi_tiến_sĩ_sau_3_năm_tốt_nghiệp_vẫn_ở_nhà_&amp;apos;ăn_bám&amp;apos;_cha__supported_summary.txt
   Số câu được xác nhận: 4
   Preview: Tô Thần Vũ, được nhận vào chương trình tiến sĩ ngành kỹ thuật y sinh tại Giang Tây, Trung Quốc, sau khi nhận được giấy báo trúng tuyển đã về quê và ở gần cha mẹ trong 3 năm. Đến năm thứ ba, khi kinh t...

📄 Du_khách_thích_thú_xem_cặp_ngựa_lùn_&amp;apos;độc_nhất_vô_nhị&amp;apos;_ở_miền_T_supported_summary.txt
   Số câu được xác nhận: 3
   Preview: Tại trại rắn Đồng Tâm ở xã Kim Sơn, tỉnh Đồng Tháp, cặp ngựa lùn pony tên Trắng và Ben 

## ✅ Hoàn thành — Bước tiếp theo

Sau khi chạy xong, hai thư mục output đã có đủ dữ liệu:

```
selected_dataset/
  ├── new_supported_summary/   ← tóm tắt đã xác nhận
  └── new_reference/           ← câu hỗ trợ từ document (.json)
```

Chạy tiếp **Bước 5**:

```bash
python structured_dataset_gen.py
```

Script này sẽ đọc `new_reference/` và sinh ra các lỗi nhân tạo trong `short_error_dataset/`.